In [3]:
import pandas as pd
import time
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

# Modeller
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Gereksiz uyarıları gizlemek için
warnings.filterwarnings('ignore')

# 1. Veriyi Yükleme
df = pd.read_csv("final_train_data.csv")

X = df.drop("Transported", axis=1)
y = df["Transported"]

# 2. Veriyi Eğitim ve Test Olarak Bölme
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Veriyi Ölçeklendirme (Tüm modelleri aynı ölçekte adil yarıştırmak için)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Modeller ve Denenecek Hiperparametreler (Sözlük Formatında)
modeller_ve_parametreler = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000),
        {'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs']}
    ),
    "Naive Bayes": (
        GaussianNB(),
        {'var_smoothing': [1e-9, 1e-8, 1e-7]} # NB için varyans yumuşatma
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {'max_depth': [None, 5, 10, 15], 'min_samples_split': [2, 5, 10]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]}
    ),
    "XGBoost": (
        XGBClassifier(random_state=42, eval_metric='logloss'),
        {'n_estimators': [50, 100, 200], 'learning_rate': [0.05, 0.1, 0.2], 'max_depth': [3, 5, 7]}
    ),
    "LightGBM": (
        LGBMClassifier(random_state=42, verbose=-1),
        {'n_estimators': [50, 100, 200], 'learning_rate': [0.05, 0.1, 0.2], 'max_depth': [3, 5, 7]}
    )
}

# 5. Tüm Modelleri GridSearch ile Optimize Edip Yarıştırma Fonksiyonu
def sampiyonlar_ligi(X_train, X_test, y_train, y_test):
    print(f"{'Model Adı':<22} | {'Süre (Sn)':<10} | {'Doğruluk':<10} | {'F1 Skor':<10} | {'En İyi Ayarlar'}")
    print("-" * 140)

    for isim, (model, parametreler) in modeller_ve_parametreler.items():
        baslangic = time.time()

        # GridSearchCV Tanımlama (cv=5 ile 5 katmanlı çapraz doğrulama yapıyoruz)
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=parametreler,
            cv=5,
            scoring='accuracy',
            n_jobs=-1 # Tüm işlemci çekirdeklerini kullan
        )

        # Modeli tüm kombinasyonlarla eğitme
        grid_search.fit(X_train, y_train)

        # En iyi modelle test verisi üzerinde tahmin yapma
        en_iyi_model = grid_search.best_estimator_
        tahmin = en_iyi_model.predict(X_test)

        # Metrikleri hesaplama
        bitis = time.time()
        sure = bitis - baslangic
        acc = accuracy_score(y_test, tahmin)
        f1 = f1_score(y_test, tahmin)

        # En iyi parametreleri metin formatına çevirme (Ekrana sığması için gereksiz süslü parantezleri atıyoruz)
        en_iyi_param_str = str(grid_search.best_params_).replace("{", "").replace("}", "")

        # Sonuçları yazdırma
        print(f"{isim:<22} | {sure:<10.2f} | {acc:<10.3f} | {f1:<10.3f} | {en_iyi_param_str}")

# 6. Yarışmayı Başlatıyoruz (Ölçeklenmiş verileri veriyoruz)
print("Modeller optimize ediliyor, bu işlem biraz sürebilir. Lütfen bekleyin...\n")
sampiyonlar_ligi(X_train_scaled, X_test_scaled, y_train, y_test)

Modeller optimize ediliyor, bu işlem biraz sürebilir. Lütfen bekleyin...

Model Adı              | Süre (Sn)  | Doğruluk   | F1 Skor    | En İyi Ayarlar
--------------------------------------------------------------------------------------------------------------------------------------------
Logistic Regression    | 3.59       | 0.740      | 0.714      | 'C': 0.01, 'solver': 'liblinear'
Naive Bayes            | 0.04       | 0.721      | 0.712      | 'var_smoothing': 1e-08
Decision Tree          | 0.38       | 0.761      | 0.744      | 'max_depth': 5, 'min_samples_split': 2
Random Forest          | 4.62       | 0.768      | 0.747      | 'max_depth': 10, 'n_estimators': 50
XGBoost                | 2.07       | 0.767      | 0.747      | 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200
LightGBM               | 34.06      | 0.770      | 0.749      | 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 50


In [5]:
import pandas as pd
import time
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

# Klasik Modeller
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Topluluk Modeli (Gizli Silah)
from sklearn.ensemble import VotingClassifier

# Gereksiz uyarıları gizlemek için
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. VERİYİ YÜKLEME VE HAZIRLAMA
# ---------------------------------------------------------
print("Veriler Yükleniyor ve Hazırlanıyor...\n")

# DİKKAT: Kaggle submission yapabilmek için 'test_algo2.csv' dosyasının ve
# ID'lerin hazır olduğunu varsayıyoruz.
# (Eğer Kaggle testi yoksa, kodun sonundaki submission kısmını silebilirsiniz).

df = pd.read_csv("final_train_data.csv")
X = df.drop("Transported", axis=1)
y = df["Transported"]

# Eğitim ve Değerlendirme (Validation) Olarak Bölme
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Veriyi Ölçeklendirme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# ---------------------------------------------------------
# 2. MODELLER VE DENENECEK HİPERPARAMETRELER
# ---------------------------------------------------------
modeller_ve_parametreler = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000),
        {'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs']}
    ),
    "Naive Bayes": (
        GaussianNB(),
        {'var_smoothing': [1e-9, 1e-8, 1e-7]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {'max_depth': [None, 5, 10, 15], 'min_samples_split': [2, 5, 10]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]}
    ),
    "XGBoost": (
        XGBClassifier(random_state=42, eval_metric='logloss'),
        {'n_estimators': [50, 100, 200], 'learning_rate': [0.05, 0.1, 0.2], 'max_depth': [3, 5, 7]}
    ),
    "LightGBM": (
        LGBMClassifier(random_state=42, verbose=-1),
        {'n_estimators': [50, 100, 200], 'learning_rate': [0.05, 0.1, 0.2], 'max_depth': [3, 5, 7]}
    )
}

# ---------------------------------------------------------
# 3. MODELLERİ OPTİMİZE EDİP YARIŞTIRMA (BİREYSEL)
# ---------------------------------------------------------
def sampiyonlar_ligi(X_train, X_val, y_train, y_val):
    print("Modeller optimize ediliyor, bu işlem biraz sürebilir. Lütfen bekleyin...\n")
    print(f"{'Model Adı':<22} | {'Süre (Sn)':<10} | {'Doğruluk':<10} | {'F1 Skor':<10}")
    print("-" * 60)

    egitilmis_en_iyi_modeller = {} # Şampiyonları Konsey için burada biriktireceğiz

    for isim, (model, parametreler) in modeller_ve_parametreler.items():
        baslangic = time.time()

        grid_search = GridSearchCV(
            estimator=model,
            param_grid=parametreler,
            cv=5,
            scoring='accuracy',
            n_jobs=-1
        )

        grid_search.fit(X_train, y_train)

        # En iyi modelle tahmin yapma
        en_iyi_model = grid_search.best_estimator_
        tahmin = en_iyi_model.predict(X_val)

        sure = time.time() - baslangic
        acc = accuracy_score(y_val, tahmin)
        f1 = f1_score(y_val, tahmin)

        # Eğitilmiş en iyi modeli sözlüğe kaydet
        egitilmis_en_iyi_modeller[isim] = en_iyi_model

        print(f"{isim:<22} | {sure:<10.2f} | {acc:<10.3f} | {f1:<10.3f}")

    return egitilmis_en_iyi_modeller

# Yarışmayı başlatıp şampiyonları alıyoruz
en_iyi_modeller = sampiyonlar_ligi(X_train_scaled, X_val_scaled, y_train, y_val)


# ---------------------------------------------------------
# 4. BÜYÜK FİNAL: ŞAMPİYONLAR KONSEYİ (VOTING CLASSIFIER)
# ---------------------------------------------------------
print("\n🚀 BÜYÜK FİNAL: ŞAMPİYONLAR KONSEYİ KURULUYOR...\n")

# GridSearch'ten başarıyla çıkmış "En İyi" modelleri Oylama Konseyine atıyoruz.
voter_modeller = [
    ('XGBoost', en_iyi_modeller["XGBoost"]),
    ('LightGBM', en_iyi_modeller["LightGBM"]),
    ('RandomForest', en_iyi_modeller["Random Forest"])
]

# Yumuşak Oylama (Soft Voting)
konsey = VotingClassifier(estimators=voter_modeller, voting='soft')

print("Konsey Ortak Akıl için eğitiliyor...")
# Zaten en iyi ayarları bulduğumuz için bu adım çok hızlı sürecektir.
konsey.fit(X_train_scaled, y_train)

# Konseyin başarısını ölçme
tahmin_konsey = konsey.predict(X_val_scaled)
skor_konsey = accuracy_score(y_val, tahmin_konsey)

print("-" * 50)
print(f"🏆 KONSEYİN (Topluluk Modeli) YENİ SKORU: {skor_konsey:.4f}")
print("-" * 50)


# ---------------------------------------------------------
# 5. KAGGLE İÇİN NİHAİ TESLİM (SUBMISSION) DOSYASINI HAZIRLAMA
# ---------------------------------------------------------
print("\nKaggle için Nihai Dosya hazırlanıyor...")

try:
    # Test verisini yüklüyoruz (Dosya adlarınızın bu şekilde olduğunu varsaydık)
    test_df = pd.read_csv("test_algo2.csv")
    test_ids = test_df["PassengerId"]

    # İhtiyacımız olmayan sütunu çıkarıp ölçeklendiriyoruz
    X_test_Kaggle = test_df.drop("PassengerId", axis=1, errors='ignore')
    X_test_Kaggle_scaled = scaler.transform(X_test_Kaggle)

    # Konsey modelini Kaggle test verisine uyguluyoruz
    kaggle_tahminleri = konsey.predict(X_test_Kaggle_scaled)
    kaggle_tahminleri_bool = [True if val == 1 else False for val in kaggle_tahminleri]

    # Dosyayı kaydetme
    submission_df = pd.DataFrame({
        "PassengerId": test_ids,
        "Transported": kaggle_tahminleri_bool
    })

    submission_df.to_csv("submission_final_konsey.csv", index=False)
    print("✅ Başarılı! 'submission_final_konsey.csv' dosyası oluştu. Kaggle'a yükleyebilirsiniz!")

except FileNotFoundError:
    print("⚠️ 'test_algo2.csv' dosyası bulunamadığı için Kaggle teslim dosyası oluşturulamadı.")
    print("Sadece model eğitimini ve konsey skorunu test etmiş oldunuz.")

Veriler Yükleniyor ve Hazırlanıyor...

Modeller optimize ediliyor, bu işlem biraz sürebilir. Lütfen bekleyin...

Model Adı              | Süre (Sn)  | Doğruluk   | F1 Skor   
------------------------------------------------------------
Logistic Regression    | 0.22       | 0.740      | 0.714     
Naive Bayes            | 0.05       | 0.721      | 0.712     
Decision Tree          | 0.24       | 0.761      | 0.744     
Random Forest          | 4.25       | 0.768      | 0.747     
XGBoost                | 1.98       | 0.767      | 0.747     
LightGBM               | 34.45      | 0.770      | 0.749     

🚀 BÜYÜK FİNAL: ŞAMPİYONLAR KONSEYİ KURULUYOR...

Konsey Ortak Akıl için eğitiliyor...
--------------------------------------------------
🏆 KONSEYİN (Topluluk Modeli) YENİ SKORU: 0.7702
--------------------------------------------------

Kaggle için Nihai Dosya hazırlanıyor...
⚠️ 'test_algo2.csv' dosyası bulunamadığı için Kaggle teslim dosyası oluşturulamadı.
Sadece model eğitimini ve kons

In [ ]:
import pandas as pd

# Oluşturduğumuz nihai dosyayı okuyalım
kontrol_df = pd.read_csv("submission_final_konsey.csv")

# Bakalım model yolculara ne karar vermiş?
print(kontrol_df['Transported'].value_counts())

In [6]:
# ---------------------------------------------------------
# 5. KAGGLE İÇİN NİHAİ TESLİM (SUBMISSION) DOSYASINI HAZIRLAMA (GARANTİLİ YÖNTEM)
# ---------------------------------------------------------
print("\nKaggle için Nihai Dosya hazırlanıyor...")

try:
    # 1. ORİJİNAL Test verisini yüklüyoruz (Bunu sadece ID'lerin sırasını korumak için kullanacağız)
    # NOT: Eğer Kaggle'ın orijinal dosyası "test.csv" ise onu okuyun.
    # test_algo2.csv sizin oluşturduğunuz bir dosyaysa ID sırası bozulmuş olabilir.
    orijinal_test_df = pd.read_csv("test.csv") # Kaggle'ın verdiği ham dosya!
    safkan_id_listesi = orijinal_test_df["PassengerId"]

    # 2. İşlem görecek test verimizi yüklüyoruz (Modelin tahmin yapacağı veri)
    islem_gorecek_test_df = pd.read_csv("final_train_data.csv")
    X_test_Kaggle = islem_gorecek_test_df.drop("PassengerId", axis=1, errors='ignore')

    # 3. Hizalama (Sütunları eğitim verisiyle eşleştirme)
    X_test_Kaggle = X_test_Kaggle.reindex(columns=X_train.columns, fill_value=0)

    # 4. Ölçeklendirme
    X_test_Kaggle_scaled = scaler.transform(X_test_Kaggle)

    # 5. Konsey ile Tahmin Yapma
    kaggle_tahminleri = konsey.predict(X_test_Kaggle_scaled)

    # 6. DİKKAT: Tahmin sayısı ile Safkan ID sayısı aynı olmak ZORUNDA!
    if len(kaggle_tahminleri) != len(safkan_id_listesi):
        print(f"KRİTİK HATA! Test verisinde satır kaybı var!")
        print(f"Orijinal yolcu sayısı: {len(safkan_id_listesi)}, Sizin tahmin sayınız: {len(kaggle_tahminleri)}")
        print("Lütfen veri hazırlama aşamasında 'drop_na' (boş satırları sil) kullanıp kullanmadığınızı kontrol edin.")
    else:
        # True/False formatına çevirme
        kaggle_tahminleri_bool = [True if val == 1 else False for val in kaggle_tahminleri]

        # 7. Nihai Dosyayı Oluşturma (Safkan ID'ler ile Tahminleri Birleştirme)
        submission_df = pd.DataFrame({
            "PassengerId": safkan_id_listesi,
            "Transported": kaggle_tahminleri_bool
        })

        # Teslim dosyasını kaydetme
        submission_df.to_csv("submission_GARANTILI.csv", index=False)

        print("\n✅ Başarılı! 'submission_GARANTILI.csv' dosyası oluştu.")
        print("Modelin Karar Dağılımı:")
        print(submission_df['Transported'].value_counts())
        print("\nŞimdi bu YENİ GARANTİLİ dosyayı Kaggle'a yükleyin. Puanın zıplaması gerekiyor!")

except FileNotFoundError:
    print("⚠️ Dosya okuma hatası! Lütfen Kaggle'ın orijinal 'test.csv' dosyasının klasörde olduğundan emin olun.")


Kaggle için Nihai Dosya hazırlanıyor...
KRİTİK HATA! Test verisinde satır kaybı var!
Orijinal yolcu sayısı: 4277, Sizin tahmin sayınız: 8659
Lütfen veri hazırlama aşamasında 'drop_na' (boş satırları sil) kullanıp kullanmadığınızı kontrol edin.
